<a href="https://colab.research.google.com/github/ettapranaycs24-beep/AI_LAB/blob/main/Simulated_Annealing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from datetime import datetime
import random, time, math
from copy import deepcopy, copy
import decimal

class Board:
    def __init__(self, queen_count=8, initial_state=None):
        self.queen_count = queen_count
        if initial_state is not None:
            self.queens = list(initial_state)
            self.queen_count = len(self.queens)
        else:
            self.reset()

    def reset(self):
        self.queens = [-1 for i in range(0, self.queen_count)]
        for i in range(0, self.queen_count):
            self.queens[i] = random.randint(0, self.queen_count - 1)

    def calculateCost(self):
        threat = 0
        for queen in range(0, self.queen_count):
            for next_queen in range(queen+1, self.queen_count):
                if self.queens[queen] == self.queens[next_queen] or abs(queen - next_queen) == abs(self.queens[queen] - self.queens[next_queen]):
                    threat += 1
        return threat

    @staticmethod
    def calculateCostWithQueens(queens):
        threat = 0
        queen_count = len(queens)
        for queen in range(0, queen_count):
            for next_queen in range(queen+1, queen_count):
                if queens[queen] == queens[next_queen] or abs(queen - next_queen) == abs(queens[queen] - queens[next_queen]):
                    threat += 1
        return threat

    @staticmethod
    def toString(queens):
        return str(queens)

    def __str__(self):
        return str(self.queens)

class SimulatedAnnealing:
    def __init__(self, board):
        self.elapsedTime = 0
        self.board = board
        self.temperature = 4000
        self.sch = 0.99
        self.startTime = datetime.now()

    def run(self):
        board = self.board
        board_queens = self.board.queens[:]
        solutionFound = False

        for k in range(0, 170000):
            self.temperature *= self.sch
            board.reset()
            successor_queens = board.queens[:]
            dw = Board.calculateCostWithQueens(successor_queens) - Board.calculateCostWithQueens(board_queens)

            try:
                exp = decimal.Decimal(decimal.Decimal(math.e) ** (decimal.Decimal(-dw) * decimal.Decimal(self.temperature)))
            except OverflowError:
                exp = decimal.Decimal('Infinity') if -dw * self.temperature > 0 else decimal.Decimal('0')

            prob = float(min(1, exp)) if dw <= 0 else 1.0

            if dw > 0 or random.uniform(0, 1) < exp:
                board_queens = successor_queens[:]
                if k % 10000 == 0 or Board.calculateCostWithQueens(board_queens) == 0:
                    print(f"Iteration {k}: Temp={self.temperature:.2f}, State={board_queens}, Cost={Board.calculateCostWithQueens(board_queens)}, Prob of Acceptance={prob:.4f}")

            if Board.calculateCostWithQueens(board_queens) == 0:
                print("\nSolution Found:")
                print(Board.toString(board_queens))
                self.elapsedTime = self.getElapsedTime()
                print("Success, Elapsed Time: %sms" % (str(self.elapsedTime)))
                solutionFound = True
                break

        if not solutionFound:
            self.elapsedTime = self.getElapsedTime()
            print("Unsuccessful, Elapsed Time: %sms" % (str(self.elapsedTime)))

        return self.elapsedTime

    def getElapsedTime(self):
        endTime = datetime.now()
        elapsedTime = (endTime - self.startTime).microseconds / 1000
        return elapsedTime

if __name__ == '__main__':
    user_input = input("Enter initial board state as a list of columns (e.g. 5,3,0,0,3,1,4,1) or leave empty for random: ").strip()
    if user_input:
        initial_state = [int(x.strip()) for x in user_input.split(',')]
        board = Board(initial_state=initial_state)
    else:
        board = Board()

    print("Initial Board State:")
    print(board)
    SimulatedAnnealing(board).run()

Enter initial board state as a list of columns (e.g. 5,3,0,0,3,1,4,1) or leave empty for random: 0,4,7,5,2,6,1,3
Initial Board State:
[0, 4, 7, 5, 2, 6, 1, 3]
Iteration 0: Temp=3960.0000, State=[6, 2, 4, 6, 0, 0, 6, 0], Cost=7, Prob of Acceptance=0.9982
Iteration 151411: Temp=0.0001, State=[5, 2, 6, 1, 7, 4, 0, 3], Cost=0, Prob of Acceptance=1.0000

Solution Found:
[5, 2, 6, 1, 7, 4, 0, 3]
Success, Elapsed Time: 589.97ms
